# OCCAM Python Interface - Analysis Workflow

This notebook demonstrates the complete OCCAM workflow using the Python bindings.
Based on OCCAM Manual v3.4.1 workflow examples.

## Typical OCCAM Analysis Steps:
1. Load data file
2. Examine basic statistics
3. Perform model search
4. Select best model(s)
5. Generate fit reports for selected models
6. Compare models

## 1. Setup and Data Loading

In [ ]:
import pyoccam
import pandas as pd
import re
from datetime import datetime

print(f"PyOCCAM version: {pyoccam.__version__}")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Configuration - modify this for your dataset
DATA_FILE = "dementia05.txt"  # Change to your data file
SEARCH_TYPE = "loopless-up"   # Options: loopless-up, full-up, disjoint-up, chain-up
SEARCH_LEVELS = 5              # Number of levels to search
SEARCH_WIDTH = 3               # Number of models to keep at each level

print(f"Data file: {DATA_FILE}")
print(f"Search configuration: {SEARCH_TYPE}, levels={SEARCH_LEVELS}, width={SEARCH_WIDTH}")

In [ ]:
# Initialize OCCAM manager
manager = pyoccam.VBMManager()

# Load data file
args = ["occam", DATA_FILE]
success = manager.init_from_command_line(args)

if success:
    print("✅ Data loaded successfully")
else:
    print(f"❌ Error: Failed to load {DATA_FILE}")
    print("Please check that the file exists and is properly formatted")

## 2. Data Exploration

In [ ]:
# Display basic statistics
print("="*60)
print("BASIC STATISTICS")
print("="*60)
print(manager.get_basic_statistics())

print(f"\nSample size: {manager.get_sample_size()}")
print(f"Has test data: {manager.has_test_data()}")

In [ ]:
# Display variable list
print("="*60)
print("VARIABLES IN DATASET")
print("="*60)

variables = manager.get_variable_list()
for i, var in enumerate(variables, 1):
    print(f"{i:2d}. {var}")

print(f"\nTotal variables: {len(variables)}")

In [ ]:
# Display available search types
print("="*60)
print("AVAILABLE SEARCH ALGORITHMS")
print("="*60)

search_types = manager.get_available_search_types()
for st in search_types:
    print(f"  • {st}")

## 3. Model Search

Perform a search through the model lattice to find the best models.
The search algorithm explores different variable combinations and relationships.

In [ ]:
# Configure report format
manager.set_report_separator(pyoccam.SPACESEP)  # Use space-separated format

# Optional: customize which columns to display
# manager.set_report_variables("ID$I, Model, Level$I, h, ddf, Alpha, Inf, %dH(DV), AIC, BIC")

print(f"Performing {SEARCH_TYPE} search...")
print(f"Levels: {SEARCH_LEVELS}, Width: {SEARCH_WIDTH}")
print("This may take a moment...\n")

In [ ]:
# Generate search report
search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=False
)

print(search_report)

In [ ]:
# Save search report to file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
search_filename = f"search_report_{timestamp}.txt"

with open(search_filename, 'w') as f:
    f.write(search_report)

print(f"Search report saved to: {search_filename}")

## 4. Model Selection

Extract and analyze the models from the search report to identify the best ones.

In [ ]:
def parse_search_report(report_text):
    """Parse search report to extract model information"""
    models = []
    lines = report_text.split('\n')
    
    # Find the table header
    header_found = False
    for i, line in enumerate(lines):
        if 'ID' in line and 'Model' in line and 'Level' in line:
            header_found = True
            continue
        
        if header_found and line.strip() and not line.startswith('--'):
            # Parse model line
            parts = line.split()
            if len(parts) >= 11 and parts[0].replace('*', '').isdigit():
                model_info = {
                    'id': int(parts[0].replace('*', '')),
                    'name': parts[1],
                    'level': int(parts[2]),
                    'bic': float(parts[10]) if len(parts) > 10 else 0,
                    'aic': float(parts[9]) if len(parts) > 9 else 0,
                    'alpha': float(parts[6]) if len(parts) > 6 else 1.0,
                    'information': float(parts[7]) if len(parts) > 7 else 0,
                    'significant': '*' in parts[0]
                }
                models.append(model_info)
    
    return models

# Parse the search report
models = parse_search_report(search_report)

if models:
    print(f"Found {len(models)} models in search report\n")
    
    # Create DataFrame for easier analysis
    df_models = pd.DataFrame(models)
    
    # Sort by BIC (lower is better)
    df_models_sorted = df_models.sort_values('bic')
    
    print("Top 5 models by BIC:")
    print(df_models_sorted[['name', 'level', 'bic', 'aic', 'alpha', 'significant']].head())
    
    # Get best model
    best_model = df_models_sorted.iloc[0]
    print(f"\n🏆 Best model (by BIC): {best_model['name']}")
    print(f"   Level: {best_model['level']}, BIC: {best_model['bic']:.2f}, Alpha: {best_model['alpha']:.4f}")
else:
    print("Could not parse models from search report")
    print("Manually specify a model name for fit report")
    best_model = {'name': 'IV:Z'}  # Default

## 5. Fit Report for Best Model

Generate a detailed fit report for the best model found in the search.

In [ ]:
# Generate fit report for best model
if 'best_model' in locals():
    model_name = best_model['name']
else:
    # Manually specify model if parsing failed
    model_name = "IV:Z"  # Change this to your model

print(f"Generating fit report for model: {model_name}")
print("="*60)

fit_report = manager.generate_fit_report(model_name)
print(fit_report)

In [ ]:
# Save fit report
fit_filename = f"fit_report_{model_name.replace(':', '_')}_{timestamp}.txt"

with open(fit_filename, 'w') as f:
    f.write(fit_report)

print(f"Fit report saved to: {fit_filename}")

## 6. Compare Multiple Models

Generate fit reports for multiple models to compare their performance.

In [ ]:
# Compare top 3 models
if 'df_models_sorted' in locals() and len(df_models_sorted) >= 3:
    print("Comparing top 3 models by BIC:")
    print("="*60)
    
    comparison_data = []
    
    for i in range(min(3, len(df_models_sorted))):
        model = df_models_sorted.iloc[i]
        model_name = model['name']
        
        # Get detailed statistics
        stats = manager.get_model_statistics(model_name)
        
        comparison_data.append({
            'Model': model_name,
            'Level': model['level'],
            'BIC': stats.get('bic', 0),
            'AIC': stats.get('aic', 0),
            'Alpha': stats.get('alpha', 1.0),
            'Information': stats.get('information', 0),
            '%Correct': stats.get('pct_correct_data', 0)
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print(df_comparison.to_string(index=False))
else:
    print("Not enough models for comparison")

## 7. Alternative Report Formats

OCCAM can generate reports in different formats for various uses.

In [ ]:
# Generate CSV format for import into other tools
print("Generating CSV format search report...")
manager.set_report_separator(pyoccam.COMMASEP)

csv_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=3,  # Fewer levels for demo
    width=3,
    include_test_data=False
)

# Show first few lines
csv_lines = csv_report.split('\n')
print("CSV Format (first 10 lines):")
for line in csv_lines[:10]:
    print(line)

# Save CSV
csv_filename = f"search_report_{timestamp}.csv"
with open(csv_filename, 'w') as f:
    # Extract just the table portion
    table_started = False
    for line in csv_lines:
        if 'ID' in line and 'Model' in line:
            table_started = True
        if table_started and line.strip():
            f.write(line + '\n')

print(f"\nCSV saved to: {csv_filename}")

# Reset to space format
manager.set_report_separator(pyoccam.SPACESEP)

## 8. Custom Model Analysis

You can also analyze specific models directly.

In [ ]:
# Analyze a custom model
# Modify this to test different variable combinations
custom_model = "IV:Z"  # Example: independence model

print(f"Analyzing custom model: {custom_model}")
print("="*60)

# Create and analyze the model
model_obj = manager.make_model(custom_model, make_fit_table=True)

if model_obj and model_obj.name:
    print(f"Model created: {model_obj.name}")
    print(f"Level: {model_obj.level}")
    print(f"Degrees of freedom: {model_obj.df}")
    print(f"Information: {model_obj.information:.4f}")
    print(f"BIC: {model_obj.bic:.2f}")
    print(f"AIC: {model_obj.aic:.2f}")
    print(f"Alpha: {model_obj.alpha:.4f}")
    print(f"Percent correct: {model_obj.pct_correct_data:.2f}%")
else:
    print(f"Could not create model: {custom_model}")
    print("Check that the model notation is correct")

## Summary

This notebook demonstrated the complete OCCAM workflow:

1. **Data Loading**: Load and validate your data file
2. **Exploration**: Examine variables and basic statistics
3. **Search**: Find the best models using various search algorithms
4. **Selection**: Identify the best model(s) based on BIC, AIC, or other criteria
5. **Fit Reports**: Generate detailed reports for selected models
6. **Comparison**: Compare multiple models
7. **Export**: Save results in various formats

### Next Steps:
- Try different search algorithms (loopless, full, disjoint, chain)
- Adjust search parameters (levels, width)
- Test specific models based on domain knowledge
- Export results for publication or further analysis

In [ ]:
# Clean up - list all generated files
import os
import glob

print("Files generated in this session:")
print("="*40)

patterns = ['search_report_*.txt', 'fit_report_*.txt', '*.csv']
for pattern in patterns:
    files = glob.glob(pattern)
    for f in files:
        size = os.path.getsize(f)
        print(f"{f} ({size:,} bytes)")